# Text Classification: Techniques and Implementation


## Steps for Text Classification

1. **Data Preparation**:
   - Cleaning text (preprocessing).
   - Tokenizing text.
   - Converting text to numerical features (vectorization).
   
2. **Feature Extraction**:
   - Bag of Words (BoW).
   - Term Frequency-Inverse Document Frequency (TF-IDF).

3. **Model Training and Evaluation**:
   - Training a classification model.
   - Evaluating model performance using metrics like accuracy, precision, recall, and F1-score.

4. **Topic Modeling**:
   - Applying unsupervised techniques like LDA to identify hidden topics.

---

## Sentiment Analysis: Logistic Regression


In [1]:
# Step 1: Import Libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Step 2: Load Data
# Example dataset: Sentiment Analysis on movie reviews
data = {
    "text": [
        "I love this movie, it was fantastic!",
        "Absolutely terrible, I hated it.",
        "The storyline was dull and boring.",
        "Great actors and brilliant direction!",
        "It was okay, not the best."
    ],
    "sentiment": [1, 0, 0, 1, 1]  # 1: Positive, 0: Negative
}
df = pd.DataFrame(data)

# Step 3: Data Preprocessing
# Lowercase and remove punctuation
df['clean_text'] = df['text'].str.lower().str.replace(r'[^\w\s]', '', regex=True)

# Step 4: Feature Extraction (TF-IDF)
vectorizer = TfidfVectorizer(max_features=10)
X = vectorizer.fit_transform(df['clean_text'])
y = df['sentiment']

# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Train Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

# Step 7: Evaluate the Model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0

Confusion Matrix:
 [[0 1]
 [0 0]]


c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

## Topic Modeling: Latent Dirichlet Allocation (LDA)

Topic modeling is an unsupervised technique that identifies hidden themes or topics in a collection of text data.

---

## Steps for LDA Implementation

1. Preprocess the text data (tokenization, stopword removal).
2. Convert text into a bag-of-words representation.
3. Train the LDA model to identify topics.
4. Visualize the topics.


In [2]:
# Step 1: Import Libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Step 2: Preprocessing (Already done in Sentiment Analysis example)
# Using the same `df['clean_text']`

# Step 3: Convert Text to Bag-of-Words
count_vectorizer = CountVectorizer(max_features=10, stop_words='english')
X_bow = count_vectorizer.fit_transform(df['clean_text'])

# Step 4: Apply LDA
lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X_bow)

# Step 5: Visualize Topics
for idx, topic in enumerate(lda.components_):
    print(f"Topic #{idx}:")
    print([count_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-5:]])


Topic #0:
['fantastic', 'actors', 'great', 'direction', 'brilliant']
Topic #1:
['best', 'dull', 'boring', 'absolutely', 'hated']


## Summary

### Sentiment Analysis:
- We trained a logistic regression model to classify text as positive or negative sentiment.
- Steps included preprocessing, feature extraction (TF-IDF), and evaluation.

### Topic Modeling:
- We applied LDA to discover hidden topics in text data.
- Bag-of-Words representation was used to extract features for LDA.


# Machine Translation: Techniques and Models

## Overview

Machine Translation (MT) is a subfield of Natural Language Processing (NLP) focused on translating text or speech from one language to another. The goal is to bridge communication barriers across languages.

### Key Techniques in Machine Translation
1. **Rule-Based Machine Translation (RBMT)**:
   - Relies on linguistic rules and dictionaries for translations.
   - Example: SYSTRAN.

2. **Statistical Machine Translation (SMT)**:
   - Uses probability models to predict the most likely translation.
   - Example: Google Translate (pre-2016).

3. **Neural Machine Translation (NMT)**:
   - Employs deep learning models to perform end-to-end translations.
   - Example: Modern Google Translate, OpenNMT, Facebook FAIR's M2M-100.

---

## Neural Machine Translation (NMT)

NMT has revolutionized the field by introducing models capable of learning context and meaning across languages. Key components of NMT include:
- **Encoder-Decoder Architecture**: Converts input sentences to intermediate representations and generates output sentences.
- **Attention Mechanism**: Allows the model to focus on relevant parts of the input during translation.
- **Sequence-to-Sequence Models**: Maps a sequence in one language to a sequence in another.

---

### Steps to Implement NMT

1. Preprocess the dataset (e.g., sentence tokenization, cleaning, and padding).
2. Create a Sequence-to-Sequence model with an encoder and decoder.
3. Integrate an attention mechanism to improve translation accuracy.
4. Train the model on parallel language datasets.
5. Evaluate the model's performance using BLEU scores.


In [3]:
# Step 1: Import Required Libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model
import numpy as np

# Step 2: Load Sample Parallel Data
# Example: English to French
data = [
    ("hello", "bonjour"),
    ("how are you?", "comment ça va?"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("goodbye", "au revoir")
]

# Prepare Vocabulary
input_texts, target_texts = zip(*data)
input_characters = sorted(set("".join(input_texts)))
target_characters = sorted(set("".join(target_texts)))

# Create Character-to-Index Mappings
input_token_index = {char: idx for idx, char in enumerate(input_characters)}
target_token_index = {char: idx for idx, char in enumerate(target_characters)}

# Hyperparameters
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

# Vectorize Data
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32")
decoder_input_data = np.zeros((len(target_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")
decoder_target_data = np.zeros((len(target_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(data):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

# Step 3: Build the Encoder-Decoder Model
# Encoder
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Combine Encoder and Decoder
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

# Step 4: Train the Model
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, batch_size=64, epochs=100, validation_split=0.2)

# Step 5: Test the Model
# Define Sampling Models for Testing
encoder_model = Model(encoder_inputs, encoder_states)
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_lstm_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)

# Function for Decoding Sequences
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_token_index["\t"]] = 1.0  # Start token
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = list(target_token_index.keys())[list(target_token_index.values()).index(sampled_token_index)]
        decoded_sentence += sampled_char
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0
        states_value = [h, c]
    return decoded_sentence

# Test with Example
test_input = "hello"
test_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype="float32")
for t, char in enumerate(test_input):
    test_seq[0, t, input_token_index[char]] = 1.0


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0179 - loss: 1.4326 - val_accuracy: 0.0714 - val_loss: 1.5815
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.1607 - loss: 1.4202 - val_accuracy: 0.0714 - val_loss: 1.5791
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.1607 - loss: 1.4097 - val_accuracy: 0.0714 - val_loss: 1.5766
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.1607 - loss: 1.3998 - val_accuracy: 0.0714 - val_loss: 1.5735
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.1607 - loss: 1.3895 - val_accuracy: 0.0714 - val_loss: 1.5691
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.1607 - loss: 1.3781 - val_accuracy: 0.0714 - val_loss: 1.5620
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.1607 - loss: 1.3643 - val_accuracy: 0.1429 - val_loss: 1.5481
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.1607 - loss: 1.3452 - val_accuracy: 0.0714 - val_

## Summary of Machine Translation Class

### What We Learned:
1. Machine Translation techniques: Rule-Based, Statistical, and Neural.
2. Building an Encoder-Decoder model for sequence-to-sequence translation.
3. Implementing a basic NMT model using LSTM for character-level translation.




# Hands-on Practice: Sequence Models for Text Analysis

## Overview

Sequence models are an integral part of Natural Language Processing (NLP), designed to process sequential data such as text, speech, or time series. In this class, we will focus on sequence models for text analysis, including sentiment analysis and text generation.

### Key Concepts in Sequence Models
1. **Recurrent Neural Networks (RNNs)**:
   - Processes input sequences one step at a time, maintaining a "hidden state" to capture dependencies in sequential data.
   
2. **Long Short-Term Memory (LSTM)**:
   - Addresses the vanishing gradient problem in RNNs by introducing memory cells that capture long-term dependencies.
   
3. **Gated Recurrent Units (GRUs)**:
   - A simplified version of LSTM with fewer parameters.

### Objectives of This Class
1. Understand the structure and functionality of sequence models.
2. Implement an LSTM-based sentiment analysis model.
3. Generate text using a trained sequence model.


In [4]:
# Step 1: Import Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Step 2: Load and Preprocess Data
# Sample data for binary sentiment analysis
data = [
    ("I love this product!", 1),
    ("This is the worst experience I've had.", 0),
    ("Absolutely fantastic service!", 1),
    ("Not worth the money.", 0),
    ("Great quality and value for the price.", 1)
]

texts, labels = zip(*data)
labels = np.array(labels)

# Tokenize and Pad Sequences
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, padding="post", maxlen=10)

# Step 3: Build the LSTM Model
model = Sequential([
    Embedding(input_dim=1000, output_dim=32, input_length=10),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
print(model.summary())

# Step 4: Train the Model
history = model.fit(padded_sequences, labels, epochs=10, batch_size=2, verbose=1)

# Step 5: Evaluate the Model
test_texts = ["The product is amazing!", "Terrible service and quality."]
test_sequences = tokenizer.texts_to_sequences(test_texts)
test_padded = pad_sequences(test_sequences, padding="post", maxlen=10)
predictions = model.predict(test_padded)

for text, prediction in zip(test_texts, predictions):
    print(f"Text: {text} - Sentiment: {'Positive' if prediction > 0.5 else 'Negative'}")


c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4000 - loss: 0.6942
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6000 - loss: 0.6913
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 0.6928
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6000 - loss: 0.6889
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6000 - loss: 0.6895
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6000 - loss: 0.6839
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6000 - loss: 0.6888
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6000 - loss: 0.6862
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 0.6793
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6000 - loss: 0.6799
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Text: The product is amazing! - Sentiment: Positive
Text: Terrible service and quality. - Sentiment: Positive


## Explanation of Sentiment Analysis Implementation

### Steps:
1. **Data Preparation**:
   - Preprocess text data by tokenizing it into numerical sequences and padding them for uniformity.
   
2. **Model Architecture**:
   - Use an embedding layer to convert words into dense vector representations.
   - Apply an LSTM layer to capture sequential dependencies.
   - Add dense layers for classification, with the final layer using a sigmoid activation for binary classification.

3. **Training**:
   - Train the model using labeled data.
   
4. **Evaluation**:
   - Predict sentiment for new text inputs.



## Next: Text Generation with Sequence Models

In [5]:
# Step 1: Prepare Data for Text Generation
text = "Deep learning is amazing and fun to learn."
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([text])
sequence_data = tokenizer.texts_to_sequences([text])[0]

# Create Input and Output Sequences
sequence_length = 5
X = []
y = []
for i in range(len(sequence_data) - sequence_length):
    X.append(sequence_data[i:i+sequence_length])
    y.append(sequence_data[i+sequence_length])

X = np.array(X)
y = np.array(y)

# Step 2: Build the LSTM Model
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=10, input_length=sequence_length),
    LSTM(64, return_sequences=False),
    Dense(len(tokenizer.word_index) + 1, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
print(model.summary())

# Step 3: Train the Model
model.fit(X, y, epochs=100, verbose=1)

# Step 4: Generate Text
def generate_text(seed_text, num_chars):
    for _ in range(num_chars):
        tokenized_input = tokenizer.texts_to_sequences([seed_text])[0]
        tokenized_input = pad_sequences([tokenized_input], maxlen=sequence_length, padding="pre")
        predicted_index = np.argmax(model.predict(tokenized_input), axis=-1)
        next_char = tokenizer.index_word[predicted_index[0]]
        seed_text += next_char
    return seed_text

# Generate New Text
seed_text = "Deep "
generated_text = generate_text(seed_text, num_chars=20)
print("Generated Text:", generated_text)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 2.9433
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.9394
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.9360
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.9326
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 2.9295
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.9259
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.9223
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 2.9182
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2.9138
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.9094
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 2.9039
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2.8988
Epoch 13/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 2.8927
Epoch 14/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 2.8860
Epoch 15/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.8789
Epoch 16/100
2

## Explanation of Text Generation Implementation

### Steps:
1. **Data Preparation**:
   - Convert text into character-level sequences for sequential prediction.
   - Split sequences into input-output pairs.

2. **Model Architecture**:
   - Embed characters into dense vector representations.
   - Use an LSTM layer to learn sequential patterns.
   - Apply a dense output layer with softmax activation for character prediction.

3. **Training**:
   - Train the model on input-output character pairs.

4. **Text Generation**:
   - Predict the next character iteratively based on a seed text and append it to generate text.

-

# Hands-on Practice: Using Named Entity Recognition (NER) for Information Extraction

## Overview

Named Entity Recognition (NER) is a sub-task of Natural Language Processing (NLP) that involves identifying and classifying named entities in text into predefined categories, such as:
- **Person Names**: "Albert Einstein"
- **Organizations**: "Google"
- **Locations**: "New York"
- **Dates and Times**: "2024-12-01"
- **Monetary Values**: "$1,000"

NER is widely used for extracting valuable structured information from unstructured text data in applications like:
1. Resume screening.
2. Customer support ticket categorization.
3. Knowledge graph construction.

---

### Objectives

1. Understand the concept and applications of NER.
2. Implement NER using Python libraries such as SpaCy.
3. Extract and visualize named entities from text data.


In [6]:
# Step 1: Install and Import Libraries
#!pip install spacy
import spacy
import subprocess
import sys

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

# Sample Text
text = """
Barack Obama was born on August 4, 1961, in Honolulu, Hawaii.
He served as the 44th President of the United States from 2009 to 2017.
He graduated from Harvard University and is married to Michelle Obama.
"""

# Apply SpaCy's NLP Pipeline
doc = nlp(text)

# Display Named Entities
print("Named Entities, their labels, and positions:")
for ent in doc.ents:
    print(f"{ent.text} ({ent.label_}) - Start: {ent.start_char}, End: {ent.end_char}")


Named Entities, their labels, and positions:
Barack Obama (PERSON) - Start: 1, End: 13
August 4, 1961 (DATE) - Start: 26, End: 40
Honolulu (GPE) - Start: 45, End: 53
Hawaii (GPE) - Start: 55, End: 61
44th (ORDINAL) - Start: 80, End: 84
the United States (GPE) - Start: 98, End: 115
2009 (DATE) - Start: 121, End: 125
Harvard University (ORG) - Start: 153, End: 171
Michelle Obama (PERSON) - Start: 190, End: 204


### Step 1: Understanding the Output
- **Named Entities**: Extracted entities like "Barack Obama" and "Harvard University."
- **Labels**:
  - `PERSON`: Individual names.
  - `ORG`: Organizations.
  - `GPE`: Geopolitical entities (e.g., countries, cities).
  - `DATE`: Dates.
- **Positions**: Character indices of the entities in the original text.

---

### Step 2: Visualizing Entities
We can use SpaCy's visualization library `displacy` to render the named entities directly in the text.


In [7]:
from spacy import displacy

# Visualize Named Entities
displacy.render(doc, style="ent", jupyter=True)


### Step 2: Key Points About Visualization
- The visualization highlights entities with their labels directly in the text.
- Different entity types are color-coded for better readability.

---

### Step 3: Customizing NER Pipeline
Suppose we want to extract only specific entity types (e.g., `PERSON` and `ORG`). We can customize the process.


In [8]:
# Extract Specific Entities
person_entities = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
org_entities = [ent.text for ent in doc.ents if ent.label_ == "ORG"]

print("Person Entities:", person_entities)
print("Organization Entities:", org_entities)



Person Entities: ['Barack Obama', 'Michelle Obama']
Organization Entities: ['Harvard University']


### Step 3: Key Observations
- We filtered named entities to include only `PERSON` and `ORG` labels.
- This is useful for domain-specific applications like building knowledge graphs or analyzing resumes.

---

### Step 4: Applying NER to a Dataset
NER can also be applied to a dataset containing multiple documents. Let’s demonstrate this with a sample dataset.


In [9]:
import pandas as pd

# Sample Dataset
data = {
    "text": [
        "Elon Musk is the CEO of SpaceX and Tesla.",
        "Tim Cook leads Apple, which is headquartered in Cupertino.",
        "Sundar Pichai is the CEO of Google."
    ]
}

df = pd.DataFrame(data)

# Apply NER to Each Text
def extract_entities(text, label):
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ == label]

df["Persons"] = df["text"].apply(lambda x: extract_entities(x, "PERSON"))
df["Organizations"] = df["text"].apply(lambda x: extract_entities(x, "ORG"))

print(df)


                                                text          Persons  \
0          Elon Musk is the CEO of SpaceX and Tesla.      [Elon Musk]   
1  Tim Cook leads Apple, which is headquartered i...       [Tim Cook]   
2                Sundar Pichai is the CEO of Google.  [Sundar Pichai]   

  Organizations  
0            []  
1       [Apple]  
2      [Google]  


### Step 4: Key Observations
- Applied NER to each document in the dataset.
- Extracted specific entities (`PERSON` and `ORG`) into new columns for structured analysis.

---

### Step 5: Extending NER for Custom Entities
For domain-specific use cases, we may need to train a custom NER model. SpaCy allows us to add custom labels and train on annotated data.

Let’s add a custom entity type `PRODUCT` to demonstrate.


In [10]:
from spacy.tokens import Span

# Add Custom Entity
doc = nlp("The new iPhone 14 was released by Apple.")
# product = Span(doc, 2, 4, label="PRODUCT")
doc.ents = [Span(doc, 2, 4, label="PRODUCT")]

# Visualize the Custom Entity
displacy.render(doc, style="ent", jupyter=True)

# Display Custom Entity
print("Custom Entities:")
for ent in doc.ents:
    print(f"{ent.text} ({ent.label_})")


Custom Entities:
iPhone 14 (PRODUCT)


## Summary

### What We Learned:
1. NER identifies and classifies entities such as `PERSON`, `ORG`, and `DATE` from unstructured text.
2. SpaCy provides built-in models and tools to extract and visualize entities.
3. Customization of NER pipelines allows for domain-specific applications like identifying product names or custom labels.

---

### Next Steps:
1. Train a custom NER model using annotated data.
2. Explore pre-trained models for multilingual NER.


# Sequence to Sequence Models for NLP

## Overview

Sequence-to-Sequence (Seq2Seq) models are a class of neural networks designed to map an input sequence to an output sequence. They are commonly used in NLP tasks such as:
- **Machine Translation**: Translating text from one language to another.
- **Text Summarization**: Condensing a document into a shorter version.
- **Dialogue Systems**: Powering chatbots and virtual assistants.
- **Question Answering**: Extracting answers from text.

### Key Components of Seq2Seq Models:
1. **Encoder**:
   - Processes the input sequence.
   - Generates a context vector summarizing the sequence.
2. **Decoder**:
   - Generates the output sequence based on the context vector.
   - Produces one token at a time.

### Enhancement: Attention Mechanism
- Focuses on relevant parts of the input sequence during decoding.
- Improves performance by allowing the model to "attend" to specific tokens.

---

## Objective

In this live session, we will:
1. Understand the architecture of Seq2Seq models.
2. Implement a Seq2Seq model for a **machine translation** task.
3. Explore how the attention mechanism improves translation accuracy.


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
import numpy as np

# Sample Parallel Data: English to French
data = [
    ("hello", "bonjour"),
    ("how are you?", "comment ça va?"),
    ("thank you", "merci"),
    ("goodbye", "au revoir"),
    ("yes", "oui"),
    ("no", "non")
]

# Separate input and target texts
input_texts, target_texts = zip(*data)

# Create character sets
input_characters = sorted(set("".join(input_texts)))
target_characters = sorted(set("".join(target_texts)))

# Character to index mapping
input_token_index = {char: i for i, char in enumerate(input_characters)}
target_token_index = {char: i for i, char in enumerate(target_characters)}

# Hyperparameters
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(text) for text in input_texts])
max_decoder_seq_length = max([len(text) for text in target_texts])

# Vectorize the data
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32")
decoder_input_data = np.zeros((len(target_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")
decoder_target_data = np.zeros((len(target_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(data):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0


### Step 1: Data Preparation

1. **Parallel Data**:
   - A dataset of sentence pairs (e.g., English and French).
2. **Character Mapping**:
   - Each character is mapped to a unique index.
3. **Vectorization**:
   - Input and output texts are converted to one-hot encoded vectors.

---

### Step 2: Build the Seq2Seq Model


In [12]:
# Encoder
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Combine Encoder and Decoder
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy")
print(model.summary())


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None, 18)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, None, 16)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 256),     │    281,600 │ input_layer_6[0]… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │    279,552 │ input_layer_7[0]… │
│                     │ 256), (None,      │            │ lstm_4[0][1],     │
│                     │ 256), (None,      │            │ lstm_4[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None, 16)  │      4,112 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 565,264 (2.16 MB)

 Trainable params: 565,264 (2.16 MB)

 Non-trainable params: 0 (0.00 B)

None


### Step 2: Seq2Seq Model Architecture

1. **Encoder**:
   - Processes input sequences and generates internal states (`state_h` and `state_c`).
   - These states summarize the input sequence as a context vector.
   
2. **Decoder**:
   - Uses the encoder's states as initial states.
   - Generates output sequences token by token.

3. **Dense Layer**:
   - Applies a softmax activation to predict the next character in the sequence.

---

### Step 3: Train the Model


In [13]:
# Train the Seq2Seq Model
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, batch_size=64, epochs=100, validation_split=0.2)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 1.5340 - val_loss: 0.3948
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 1.5248 - val_loss: 0.3943
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.5153 - val_loss: 0.3937
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.5052 - val_loss: 0.3930
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.4940 - val_loss: 0.3920
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.4808 - val_loss: 0.3907
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 1.4644 - val_loss: 0.3888
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.4428 - val_loss: 0.3862
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 1.4136 - val_loss: 0.3826
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 1.3771 - val_loss: 0.3782
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 1.3538 - val_loss: 0.3734
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 1.3544

### Step 3: Training

- Train the Seq2Seq model on parallel sentence pairs.
- Use `categorical_crossentropy` as the loss function, as this is a multi-class classification problem (predicting the next token).

---

### Step 4: Translate New Sentences


In [14]:
# Define Sampling Models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_lstm_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_lstm_outputs)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)

# Function to Decode Sequences
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Instead of using '\t', use the start character from target_characters if it exists,
    # otherwise, use a space character as a default.
    start_char = '\t' if '\t' in target_characters else ' '
    target_seq[0, 0, target_token_index[start_char]] = 1.0  # Start token

    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = list(target_token_index.keys())[list(target_token_index.values()).index(sampled_token_index)]
        decoded_sentence += sampled_char

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0
        states_value = [h, c]
    return decoded_sentence

# Translate a New Sentence
test_input = "hello"
test_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype="float32")
for t, char in enumerate(test_input):
    test_seq[0, t, input_token_index[char]] = 1.0
print("Translation:", decode_sequence(test_seq))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Translation: onjourr        


### Step 4: Translation

1. **Encoder Model**:
   - Encodes the input sequence into a context vector.
   
2. **Decoder Model**:
   - Decodes the context vector to generate the output sequence.
   
3. **Prediction Loop**:
   - Generates one token at a time until the end-of-sequence token is produced.

---

## Summary

1. Implemented a Seq2Seq model for machine translation.
2. Explored the architecture with an encoder and decoder.
3. Translated new sentences using the trained model.

---

### Next Steps:
2. Train on larger datasets for real-world applications.
